In [13]:
import pandas as pd
from bs4 import BeautifulSoup as bs
import requests
import numpy as np
from playwright.async_api import async_playwright
from datetime import date
import os
from dotenv import load_dotenv

In [14]:
url = "https://daigler25.addu.edu.ph/login/index.php"
load_dotenv()
username = os.getenv("USERNAME_KEY")
password = os.getenv("PASSWORD_KEY")

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=False)
    page = await browser.new_page()
    await page.goto(url)
    
    #login
    await page.locator("#username").fill(username)
    await page.locator("#password").fill(password)
    await page.click("#loginbtn")
    
    #data
    data = []
    
    #extraction
    container = page.locator("div.calendarwrapper")
    for card in await container.locator("div.event > div.overflow-auto").all():
        title = await card.locator("h4.d-flex > a.text-truncate").inner_text()
        dates = await card.locator("div.date > a").get_attribute("title")
        data.append({
            "title" : title,
            "dates" : dates
        })
    
    
    #check
    #await page.wait_for_url("**/my/")
    
    #clos
    await browser.close()

In [15]:
df = pd.DataFrame(data)
df["title"] = df["title"].replace(r"\bis due\b", "",regex=True)
df.iloc[4]

title    Alternative NSTP Photo Essay: National Securit...
dates                                   Monday, 26 October
Name: 4, dtype: str

In [16]:
markdown_df = df.to_markdown(index=False, colalign=("left", "right"))
markdown_df = markdown_df[226:]

In [17]:
formatted_markdown = markdown_df.split("\n")

formatted = [f"{row}\n\n" for row in formatted_markdown]
formatted = "\n".join(formatted)
formatted = formatted + "\n---------------------------------"
print(formatted)

| Exercise W1101 submission                                                            | Wednesday, 21 October |


| Exercise W1102 submission                                                            | Wednesday, 21 October |


| Exercise W1103 submission                                                            | Wednesday, 21 October |


| NSTP Photo Essay: National Security Forum                                            |  Saturday, 24 October |


| Alternative NSTP Photo Essay: National Security Forum (for Students Who Were Absent) |    Monday, 26 October |


| Exercise W1201 submission                                                            | Wednesday, 28 October |


| Exercise W1202 submission                                                            | Wednesday, 28 October |


| Exercise W1203 submission                                                            | Wednesday, 28 October |


---------------------------------


In [18]:
#The notification sender

topic = "daigler-angeles-deadlines-college"
url = f"https://ntfy.sh/{topic}"
timedate = date.today()

post = requests.post(
    url,
    data=formatted,
    headers={
        "Title" : f"Daigler Deadlines as of {timedate}"
    }
)